# 01 - Data Ingestion & Merging

Loads and normalizes raw source files from `datasets/` into a canonical catalog saved at `data/catalog.csv`.

**Canonical schema:** `isbn13, title, authors, categories, description, published_year, average_rating, num_pages, ratings_count, thumbnail, source`.


In [1]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import re

import pandas as pd

from app.config import CANONICAL_COLUMNS, CATALOG_CSV, DATASETS_DIR, get_logger

logger = get_logger("ingestion")

## 1. Load and Normalize Each Source

Maps each source onto `CANONICAL_COLUMNS` and assigns a `source` tag.


In [2]:
def load_7k_books(path: Path) -> pd.DataFrame:
    """Load 7k_books.csv - its columns already match the canonical names."""
    df = pd.read_csv(path, dtype={"isbn13": "string"})
    df["source"] = "7k_books"
    return df[CANONICAL_COLUMNS + ["source"]]


books_7k = load_7k_books(DATASETS_DIR / "7k_books.csv")
logger.info("Loaded %d rows from 7k_books.csv", len(books_7k))

2026-08-17 18:26:04 | INFO    | booklens.ingestion | Loaded 6810 rows from 7k_books.csv


In [3]:
def load_google_books(path: Path) -> pd.DataFrame:
    """
    Load google_books_dataset.csv onto the canonical schema.

    This source uses different column names (book_id, page_count, isbn_13, ...)
    and a full publication date instead of a bare year, so each field needs an
    explicit mapping rather than a rename.
    """
    df = pd.read_csv(path, dtype={"isbn_13": "string"}, low_memory=False)

    published_year = pd.to_datetime(df["published_date"], errors="coerce").dt.year

    mapped = pd.DataFrame({
        "isbn13": df["isbn_13"],
        "title": df["title"],
        "authors": df["authors"],
        "categories": df["categories"],
        "description": df["description"],
        "published_year": published_year,
        "average_rating": df["average_rating"],
        "num_pages": df["page_count"],
        "ratings_count": df["ratings_count"],
        "thumbnail": df["thumbnail"],
    })
    mapped["source"] = "google_books"
    return mapped[CANONICAL_COLUMNS + ["source"]]


google_books = load_google_books(DATASETS_DIR / "google_books_dataset.csv")
logger.info("Loaded %d rows from google_books_dataset.csv", len(google_books))

2026-08-17 18:26:04 | INFO    | booklens.ingestion | Loaded 15147 rows from google_books_dataset.csv


In [4]:
def load_nepali_books(path: Path) -> pd.DataFrame:
    """
    Load nepali_books_clean.csv onto the canonical schema.

    This is the Nepali source feeding the `Nepali Literature` category -
    it already has every canonical column except thumbnail, which it
    doesn't provide.
    """
    df = pd.read_csv(path, dtype={"isbn13": "string"})
    df["thumbnail"] = pd.NA
    df["source"] = "nepali_books_clean"
    return df[CANONICAL_COLUMNS + ["source"]]


nepali_books = load_nepali_books(DATASETS_DIR / "nepali_books_clean.csv")
logger.info("Loaded %d rows from nepali_books_clean.csv", len(nepali_books))

2026-08-17 18:26:04 | INFO    | booklens.ingestion | Loaded 2495 rows from nepali_books_clean.csv


## 2. Merge Sources

Concatenates all normalized source DataFrames.


In [5]:
sources = [books_7k, google_books, nepali_books]

catalog = pd.concat(sources, ignore_index=True)
logger.info("Merged catalog before de-duplication: %d rows", len(catalog))

2026-08-17 18:26:04 | INFO    | booklens.ingestion | Merged catalog before de-duplication: 24452 rows


## 3. Deduplication

Deduplicates in two stages: first by valid ISBN-13, then by normalized (title, author).


In [6]:
def normalize_text(value) -> str:
    """Lowercase, strip punctuation and extra whitespace - used to build a dedup key."""
    if pd.isna(value):
        return ""
    value = str(value).lower().strip()
    value = re.sub(r"[^a-z0-9\s]", "", value)
    value = re.sub(r"\s+", " ", value)
    return value


def first_author(authors) -> str:
    """Take the first listed author, so 'A;B' and 'A' still match on author."""
    if pd.isna(authors):
        return ""
    return str(authors).split(";")[0].split(",")[0].strip()


catalog["_title_key"] = catalog["title"].map(normalize_text)
catalog["_author_key"] = catalog["authors"].map(first_author).map(normalize_text)
isbn_str = catalog["isbn13"].astype("string").str.strip()
catalog["_isbn_key"] = isbn_str.where(isbn_str.notna() & (isbn_str != ""))

rows_before_dedup = len(catalog)

In [7]:
# Pass 1: drop rows that share a non-empty isbn13, keeping the first occurrence.
has_isbn = catalog["_isbn_key"].notna()
catalog = pd.concat([
    catalog[has_isbn].drop_duplicates(subset="_isbn_key", keep="first"),
    catalog[~has_isbn],
], ignore_index=True)

rows_after_isbn_dedup = len(catalog)
logger.info("Pass 1 (isbn13): dropped %d duplicate rows", rows_before_dedup - rows_after_isbn_dedup)

2026-08-17 18:26:04 | INFO    | booklens.ingestion | Pass 1 (isbn13): dropped 60 duplicate rows


In [8]:
# Pass 2: drop title+author duplicates among what's left. Rows with a blank
# title key are excluded from this check entirely, so they don't all collapse
# into a single "duplicate" group just for sharing an empty string.
has_title = catalog["_title_key"] != ""
is_title_author_dupe = has_title & catalog.duplicated(subset=["_title_key", "_author_key"], keep="first")
catalog = catalog[~is_title_author_dupe].reset_index(drop=True)

rows_after_title_dedup = len(catalog)
logger.info("Pass 2 (title+author): dropped %d duplicate rows", rows_after_isbn_dedup - rows_after_title_dedup)

catalog = catalog.drop(columns=["_title_key", "_author_key", "_isbn_key"])
logger.info(
    "Catalog after de-duplication: %d rows (dropped %d total, %.1f%%)",
    len(catalog), rows_before_dedup - len(catalog),
    100 * (rows_before_dedup - len(catalog)) / rows_before_dedup,
)

2026-08-17 18:26:04 | INFO    | booklens.ingestion | Pass 2 (title+author): dropped 1816 duplicate rows
2026-08-17 18:26:04 | INFO    | booklens.ingestion | Catalog after de-duplication: 22576 rows (dropped 1876 total, 7.7%)


## 4. Cleaning & Validation

Strips whitespace, coerces numeric fields, and drops entries missing required titles.


In [9]:
text_columns = ["title", "authors", "categories", "description", "thumbnail"]
for col in text_columns:
    catalog[col] = catalog[col].astype("string").str.strip()

numeric_columns = ["published_year", "average_rating", "num_pages", "ratings_count"]
for col in numeric_columns:
    catalog[col] = pd.to_numeric(catalog[col], errors="coerce")

In [10]:
missing_title = catalog["title"].isna() | (catalog["title"] == "")
if missing_title.any():
    logger.warning("Dropping %d rows with no title", missing_title.sum())
    catalog = catalog[~missing_title].reset_index(drop=True)

missing_description = catalog["description"].isna() | (catalog["description"] == "")
logger.warning(
    "%d of %d rows (%.1f%%) have no description - still embedded from "
    "title/authors/categories, just with a weaker search signal",
    missing_description.sum(), len(catalog), 100 * missing_description.mean(),
)

2026-08-17 18:26:04 | WARNING | booklens.ingestion | Dropping 8 rows with no title
2026-08-17 18:26:04 | WARNING | booklens.ingestion | 6083 of 22568 rows (27.0%) have no description - still embedded from title/authors/categories, just with a weaker search signal


## 5. Save Canonical Catalog

Exports the unified dataset to `data/catalog.csv` for downstream indexing and modeling.


In [11]:
catalog = catalog[CANONICAL_COLUMNS + ["source"]]

CATALOG_CSV.parent.mkdir(parents=True, exist_ok=True)
catalog.to_csv(CATALOG_CSV, index=False)

logger.info("Saved merged catalog to %s (%d rows, %d columns)", CATALOG_CSV, *catalog.shape)
logger.info("Rows per source:\n%s", catalog["source"].value_counts().to_string())
catalog.head()

2026-08-17 18:26:05 | INFO    | booklens.ingestion | Saved merged catalog to /Users/obscure/Developer/MINOR/book_lens/data/catalog.csv (22568 rows, 11 columns)
2026-08-17 18:26:05 | INFO    | booklens.ingestion | Rows per source:
source
google_books          13597
7k_books               6568
nepali_books_clean     2403


,isbn13,title,authors,categories,description,published_year,average_rating,num_pages,ratings_count,thumbnail,source
0,9780002005883,Gilead,Marilynne Robinson,Fiction,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,http://books.google.com/books/content?id=KQZCP...,7k_books
1,9780002261982,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,http://books.google.com/books/content?id=gA5GP...,7k_books
2,9780006163831,The One Tree,Stephen R. Donaldson,American fiction,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0,http://books.google.com/books/content?id=OmQaw...,7k_books
3,9780006178736,Rage of angels,Sidney Sheldon,Fiction,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,http://books.google.com/books/content?id=FKo2T...,7k_books
4,9780006280897,The Four Loves,Clive Staples Lewis,Christian life,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,http://books.google.com/books/content?id=XhQ5X...,7k_books
